# Heaps · Study Notes

KTH  
Sep 08, 2026

---
> Using F-heaps we are able to obtain improved running times for several network
> optimization algorithms.
>
> — "Fibonacci heaps and their uses," M. L. Fredman and R. E. Tarjan, 1987

These notes cover the chapter's conceptual material: heap fundamentals and the
array representation, the heaps boot camp (`top_k` over a stream), the Top Tips
table, and the `heapq` library — including the max-heap workaround. Code cells
are runnable.

## 1. Heap fundamentals

A **heap** is a specialized binary tree. Specifically, it is a **complete binary
tree** (every level except possibly the last is completely filled, and all nodes
are as far left as possible — see Chapter 6). The keys must satisfy the **heap
property**:

> The key at each node is **at least as great as** the keys stored at its
> children.

That describes a **max-heap**. The **min-heap** is a completely symmetric version
of the data structure, supporting `O(1)` time lookups for the **minimum**
element.

### Complexity

| Operation | Time |
|---|---|
| Insertion | `O(log n)` |
| Lookup of the max element | `O(1)` |
| Deletion of the max element | `O(log n)` |
| **Search for an arbitrary key** | **`O(n)`** |

The **extract-max** operation is defined to delete *and return* the maximum
element.

### Priority queue

A heap is sometimes referred to as a **priority queue** because it behaves like
a queue, with one difference: each element has a "priority" associated with it,
and **deletion removes the element with the highest priority**.

## 2. The array representation

A max-heap can be implemented as an **array**: the children of the node at index
*i* are at indices **2*i* + 1** and **2*i* + 2**.

The array representation for the max-heap in Figure 7.1(a) is:

⟨561, 314, 401, 28, 156, 359, 271, 11, 3⟩

```
                    561
              /            \
          314              401
challenge  /    \          /    \
        28      156     359     271
       /  \
     11    3
```

**Deletion of the max** (Figure 7.1(b)) is performed by **replacing the root's
key with the key at the last leaf**, and then recovering the heap property by
**repeatedly exchanging keys with children**. Deleting 561 from the heap above
leaves ⟨401, 314, 359, 28, 156, 3, 271, 11⟩ with 401 at the root.

In [ ]:
# The Figure 7.1(a) max-heap, and a check that the heap property holds.
H = [561, 314, 401, 28, 156, 359, 271, 11, 3]

def children(i, n):
    '''Indices of the children of node i, per the 2i+1 / 2i+2 rule.'''
    return [c for c in (2 * i + 1, 2 * i + 2) if c < n]

def is_max_heap(A):
    return all(A[i] >= A[c] for i in range(len(A)) for c in children(i, len(A)))

print("array :", H)
print("root (max) =", H[0], "-> O(1) lookup")
print("is a valid max-heap:", is_max_heap(H))

print()
for i, v in enumerate(H):
    kids = [H[c] for c in children(i, len(H))]
    print(f"  index {i}: {v:>3}  children -> {kids}")

In [ ]:
# Deletion of the max, exactly as Figure 7.1(b) describes it.
def sift_down(A, i):
    n = len(A)
    while True:
        largest = i
        for c in children(i, n):
            if A[c] > A[largest]:
                largest = c
        if largest == i:
            return
        A[i], A[largest] = A[largest], A[i]     # exchange with the larger child
        i = largest

def extract_max(A):
    max_value = A[0]
    A[0] = A[-1]        # replace root's key with the key at the last leaf
    A.pop()
    if A:
        sift_down(A, 0)  # recover the heap property
    return max_value

h = list(H)
print("before      :", h)
print("extract_max :", extract_max(h))
print("after       :", h)
print("still valid :", is_max_heap(h), "| new max:", h[0])

## 3. Heaps boot camp — the *k* longest strings in a stream

**Problem.** Write a program that takes a sequence of strings presented in
**"streaming" fashion** — you cannot back up to read an earlier value. The
program must compute the **k longest strings** in the sequence. Only the *k*
longest strings are required; it is **not** required to order them.

**Key insight.** As we process the input, we want to track the *k* longest
strings seen so far. Out of these *k* strings, the one to be **evicted** when a
longer string arrives is the **shortest** one.

So a **min-heap (not a max-heap!)** is the right structure here, since it
supports efficient find-min, remove-min, and insert. The program uses a heap
with a custom comparison, where strings are ordered by **length**.

In [ ]:
import heapq
import itertools

def top_k(k, stream):
    # Entries are compared by their lengths.
    min_heap = [(len(s), s) for s in itertools.islice(stream, k)]
    heapq.heapify(min_heap)
    for next_string in stream:
        # Push next_string and pop the shortest string in min_heap.
        heapq.heappushpop(min_heap, (len(next_string), next_string))
    return [p[1] for p in heapq.nsmallest(k, min_heap)]

In [ ]:
words = ["a", "bb", "ccc", "dddd", "ee", "ffffff", "g", "hhhhh"]

# `stream` must be an ITERATOR, not a list — islice consumes the first k,
# then the for-loop continues from where islice stopped.
print(top_k(3, iter(words)))

> ⚠️ **A subtlety in the boot camp code.** `top_k` relies on `stream` being a
> single-pass **iterator**. `itertools.islice` consumes the first *k* items, and
> the `for` loop then resumes from item *k+1*. If you pass a **list** instead,
> the `for` loop restarts from the beginning and the first *k* strings get
> processed twice. That's fine for this problem's correctness (duplicates of
> already-seen strings can only be re-inserted, not lost) but it's the kind of
> detail worth noticing — the "streaming" framing is what makes the iterator
> assumption natural.

In [ ]:
# The difference, made visible: pick a case where re-reading the head matters.
short_first = ["zzzz", "yyy", "a", "bb"]     # the two longest are at the FRONT

print("iterator (correct)   :", sorted(top_k(2, iter(short_first))))
print("list (head re-read)  :", sorted(top_k(2, short_first)))
print()
print("With a list, islice takes ['zzzz','yyy'] AND the for-loop starts over at")
print("'zzzz', so early items get processed twice. Here the result survives, but")
print("the heap does strictly more work than the O(n log k) analysis assumes.")

**Complexity.** Each string is processed in **`O(log k)`** time — the time to
add and remove the minimum element from the heap. So if there are *n* strings in
the input, the total time complexity is **`O(n log k)`**.

**A best-case improvement.** We could improve best-case time by first comparing
the new string's length with the length of the string at the **top of the heap**
(getting this takes `O(1)` time) and **skipping the insert** if the new string is
too short to be in the set.

In [ ]:
def top_k_optimized(k, stream):
    '''Same result, but skips the O(log k) insert when the candidate can't qualify.'''
    min_heap = [(len(s), s) for s in itertools.islice(stream, k)]
    heapq.heapify(min_heap)
    for next_string in stream:
        if len(next_string) > min_heap[0][0]:      # O(1) peek at the smallest
            heapq.heappushpop(min_heap, (len(next_string), next_string))
        # else: too short to ever qualify — skip the insert entirely
    return [p[1] for p in heapq.nsmallest(k, min_heap)]

print(top_k_optimized(3, iter(words)))

## 4. Table 7.1 — Top Tips for Heaps

- Use a heap when **all you care about is the largest or smallest elements**,
  and you do **not** need to support fast lookup, delete, or search operations
  for arbitrary elements.
- A heap is a good choice when you need to compute the **k largest** or **k
  smallest** elements in a collection. For the former, use a **min-heap**; for
  the latter, use a **max-heap**.

> 💡 **Why the inversion?** It reads backwards at first. To keep the *k largest*
> items, you hold them in a **min-heap** so the root is the **weakest survivor**
> — the one to evict when something bigger arrives. Symmetrically for *k
> smallest* with a max-heap. This is exactly the boot camp's "not a max-heap!"
> parenthetical, and it's the single most commonly reversed heap decision in
> interviews.

## 5. Know your heap libraries

Heap functionality in Python is provided by the **`heapq`** module.

| Function | Meaning |
|---|---|
| `heapq.heapify(L)` | Transforms the elements in `L` into a heap **in-place**. |
| `heapq.nlargest(k, L)` / `heapq.nsmallest(k, L)` | Returns the *k* largest / smallest elements in `L`. |
| `heapq.heappush(h, e)` | Pushes a new element on the heap. |
| `heapq.heappop(h)` | Pops the **smallest** element from the heap. |
| `heapq.heappushpop(h, a)` | Pushes `a` on the heap, then pops and returns the smallest element. |
| `e = h[0]` | Returns the smallest element **without popping it**. |

### ⚠️ The single most important `heapq` fact

**`heapq` only provides min-heap functionality.** If you need a **max-heap** on
integers or floats, **insert their negatives** to get the max-heap effect. For
objects, implement `__lt__` appropriately.

In [ ]:
import heapq

L = [561, 314, 401, 28, 156, 359, 271, 11, 3]

h = list(L)
heapq.heapify(h)                     # in-place; now a MIN-heap
print("heapify     :", h)
print("h[0] (peek) :", h[0], "-> the SMALLEST, not the largest")
print("nlargest(3) :", heapq.nlargest(3, L))
print("nsmallest(3):", heapq.nsmallest(3, L))

heapq.heappush(h, 100)
print("after push  :", heapq.heappop(h), "popped (smallest)")
print("pushpop 0   :", heapq.heappushpop(h, 0), "<- pushes 0, immediately pops it back")

In [ ]:
# The negation trick: a max-heap on numbers, using heapq
max_heap = []
for x in L:
    heapq.heappush(max_heap, -x)     # store negatives

print("largest  :", -max_heap[0])                 # negate on the way out
print("extract  :", -heapq.heappop(max_heap))
print("next     :", -max_heap[0])

In [ ]:
import functools

# For OBJECTS, implement __lt__ instead of negating.
@functools.total_ordering
class Task:
    def __init__(self, name, priority):
        self.name, self.priority = name, priority

    def __lt__(self, other):
        # Reverse the comparison to make heapq behave as a MAX-heap on priority.
        return self.priority > other.priority

    def __eq__(self, other):
        return self.priority == other.priority

    def __repr__(self):
        return f"{self.name}({self.priority})"

tasks = []
for t in [Task('email', 2), Task('outage', 9), Task('lunch', 1), Task('review', 5)]:
    heapq.heappush(tasks, t)

print("highest priority first:", [heapq.heappop(tasks) for _ in range(4)])

## 6. Big picture

A heap trades away everything a BST or hash table gives you — ordered iteration,
successor/predecessor, fast arbitrary lookup — in exchange for making **one**
question cheap: *what is the extreme element?* `O(1)` to look at it, `O(log n)`
to remove it, `O(log n)` to insert.

That narrow contract is why the "k largest → min-heap" inversion works: you only
ever need to know the weakest thing you're currently keeping, and a heap answers
exactly that in constant time. When a problem says "top k," "k closest,"
"running median," or "merge k sorted things," the heap is usually the intended
structure.